# SENSERO – Stage 2: Attach metadata to dataset GeoPackage

Reads the Stage 1 GeoPackage (`sensero_patches.gpkg`) and `metadata_336.parquet`,
joins on filename (stripping the `.tif` suffix from the GeoPackage side), and writes
the final **`sensero.gpkg`**.

## What the notebook does

1. Loads `sensero_patches.gpkg` (Stage 1) — keeps only `filename`, `rel_path`, and `geometry`
2. Loads `metadata_336.parquet` with captions, CLC codes, and split assignment
3. Strips `.tif` from the GeoPackage `filename` column so it matches `BaseFilename`
4. Validates the join (overlap / unmatched / duplicates)
5. Merges and **hard-asserts zero unmatched rows**
6. Writes `sensero.gpkg`
7. Round-trip verifies attribute count, types, and a sample feature


In [1]:
# ============================================================================
# CONFIG
# ============================================================================
from pathlib import Path

GPKG_IN     = Path("/home/ubuntu/SENSERO/sensero_patches.gpkg")   # Stage 1 output
PARQUET_IN  = Path("/home/ubuntu/SENSERO/GeoTiff/Patch_336/metadata_336.parquet")
GPKG_OUT    = Path("/home/ubuntu/SENSERO/sensero.gpkg")
LAYER_NAME  = "patches"   # GeoPackage can hold multiple layers; we use one

# Columns to keep from the Stage 1 GeoPackage (besides geometry)
KEEP_FROM_GPKG = ["filename", "rel_path"]

# Columns to bring across from the parquet
NEW_COLS = ["caption1", "caption2", "caption3", "caption4", "caption5",
            "CLC_codes", "split"]

# Keep filename without ".tif" after merge?
#   True  → output column is bare stem (matches parquet, easier to re-join later)
#   False → re-append ".tif" after merge so downstream code that expects the
#           extension still works
STRIP_TIF_IN_OUTPUT = True

print(f"Input GeoPackage : {GPKG_IN}")
print(f"Input parquet    : {PARQUET_IN}")
print(f"Output GeoPackage: {GPKG_OUT}")
print(f"Layer name       : {LAYER_NAME}")
print(f"Keep from GPKG   : {KEEP_FROM_GPKG}")
print(f"New columns      : {NEW_COLS}")
print(f"Strip .tif       : {STRIP_TIF_IN_OUTPUT}")


Input GeoPackage : /home/ubuntu/SENSERO/sensero_patches.gpkg
Input parquet    : /home/ubuntu/SENSERO/GeoTiff/Patch_336/metadata_336.parquet
Output GeoPackage: /home/ubuntu/SENSERO/sensero.gpkg
Layer name       : patches
Keep from GPKG   : ['filename', 'rel_path']
New columns      : ['caption1', 'caption2', 'caption3', 'caption4', 'caption5', 'CLC_codes', 'split']
Strip .tif       : True


In [2]:
# ============================================================================
# LOAD
# ============================================================================
import geopandas as gpd
import pandas as pd

gdf_raw = gpd.read_file(GPKG_IN)
print(f"GeoPackage (raw) : {len(gdf_raw):,} features, CRS = {gdf_raw.crs}")
print(f"  all columns    : {list(gdf_raw.columns)}")

# Keep only the columns we need + geometry
gdf = gdf_raw[KEEP_FROM_GPKG + ["geometry"]].copy()
print(f"  kept columns   : {list(gdf.columns)}")

df = pd.read_parquet(PARQUET_IN)
print(f"\nParquet          : {len(df):,} rows")
print(f"  columns        : {list(df.columns)}")

missing_in_parquet = [c for c in NEW_COLS + ["BaseFilename"] if c not in df.columns]
if missing_in_parquet:
    raise ValueError(f"Parquet missing expected columns: {missing_in_parquet}")


GeoPackage (raw) : 10,000 features, CRS = EPSG:4326
  all columns    : ['filename', 'rel_path', 'subfolder', 'width_px', 'height_px', 'bands', 'crs_src', 'area_m2', 'geometry']
  kept columns   : ['filename', 'rel_path', 'geometry']

Parquet          : 10,000 rows
  columns        : ['BaseFolder', 'BaseFilename', 'CLC_codes', 'caption1', 'caption2', 'caption3', 'caption4', 'caption5', 'split']


ERROR 1: libpoppler.so.140: cannot open shared object file: No such file or directory
ERROR 1: libpoppler.so.140: cannot open shared object file: No such file or directory


In [3]:
# ============================================================================
# FIX JOIN KEY — strip .tif from GeoPackage filenames
# ============================================================================
# GeoPackage filenames end in ".tif"; parquet BaseFilename does not.
# Strip the extension on the GeoPackage side so the merge can work.

print("Before strip — sample GeoPackage filenames:")
for f in gdf["filename"].head(3):
    print(f"  {f!r}")

gdf["filename"] = gdf["filename"].str.removesuffix(".tif")

print("\nAfter strip — sample GeoPackage filenames:")
for f in gdf["filename"].head(3):
    print(f"  {f!r}")

print("\nSample parquet BaseFilenames (for comparison):")
for f in df["BaseFilename"].head(3):
    print(f"  {f!r}")


Before strip — sample GeoPackage filenames:
  'S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10056_3577.tif'
  'S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10062_1155.tif'
  'S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10093_8416.tif'

After strip — sample GeoPackage filenames:
  'S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10056_3577'
  'S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10062_1155'
  'S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10093_8416'

Sample parquet BaseFilenames (for comparison):
  'S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM_1002_3034'
  'S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM_1049_1798'
  'S2A_MSIL2A_20180809T090551_N0500_R050_T35TNM_1071_6708'


In [4]:
# ============================================================================
# VALIDATE JOIN KEYS
# ============================================================================
gpkg_keys = set(gdf["filename"])
pq_keys   = set(df["BaseFilename"])

in_both   = gpkg_keys & pq_keys
gpkg_only = gpkg_keys - pq_keys
pq_only   = pq_keys  - gpkg_keys

print(f"GeoPackage unique filenames : {len(gpkg_keys):,}")
print(f"Parquet    unique filenames : {len(pq_keys):,}")
print(f"In both                     : {len(in_both):,}")
print(f"GeoPackage only (unmatched) : {len(gpkg_only):,}")
print(f"Parquet only    (unused)    : {len(pq_only):,}")

if len(in_both) == 0:
    raise ValueError(
        "Zero overlap between GeoPackage and parquet keys — the join would produce "
        "all-NaN attributes. Check that .tif stripping worked and that both "
        "datasets refer to the same patch size."
    )

# Check for duplicates on either side (would break one-to-one merge)
dup_gpkg = gdf["filename"].duplicated().sum()
dup_pq   = df["BaseFilename"].duplicated().sum()
print(f"\nDuplicate keys in GeoPackage: {dup_gpkg}")
print(f"Duplicate keys in parquet   : {dup_pq}")
if dup_gpkg or dup_pq:
    raise ValueError("Duplicate join keys found — one-to-one merge will fail.")

if gpkg_only:
    print(f"\n⚠  {len(gpkg_only)} GeoPackage features have no parquet entry — "
          "they will get NaN attributes (merge will then fail the hard check below).")

if pq_only:
    print(f"\nℹ  {len(pq_only)} parquet rows have no GeoPackage feature — "
          "harmless (they're just unused).")


GeoPackage unique filenames : 10,000
Parquet    unique filenames : 10,000
In both                     : 10,000
GeoPackage only (unmatched) : 0
Parquet only    (unused)    : 0

Duplicate keys in GeoPackage: 0
Duplicate keys in parquet   : 0


In [5]:
# ============================================================================
# MERGE + HARD CHECK
# ============================================================================
# Bring across only what we need, plus the join key
to_attach = df[["BaseFilename"] + NEW_COLS].copy()

# CLC_codes may be a list/array in parquet — coerce to string for safe storage.
# GeoPackage handles arbitrary text length, so no truncation worry.
to_attach["CLC_codes"] = to_attach["CLC_codes"].apply(
    lambda v: ", ".join(map(str, v)) if hasattr(v, "__iter__") and not isinstance(v, str) else str(v)
)

merged = gdf.merge(
    to_attach,
    left_on="filename",
    right_on="BaseFilename",
    how="left",
    validate="one_to_one",
)
merged = merged.drop(columns=["BaseFilename"])

# Hard check: every feature must have attributes attached.
unmatched = merged[NEW_COLS[0]].isna().sum()
if unmatched > 0:
    raise ValueError(
        f"{unmatched:,}/{len(merged):,} features failed to merge. "
        "Join key mismatch — inspect filename vs BaseFilename columns."
    )

# Optional: re-append .tif to the filename column for downstream consumers
if not STRIP_TIF_IN_OUTPUT:
    merged["filename"] = merged["filename"] + ".tif"

print(f"Merged : {len(merged):,} features")
print(f"Columns: {list(merged.columns)}")
print(f"All {len(merged):,} features have attributes ✓")


Merged : 10,000 features
Columns: ['filename', 'rel_path', 'geometry', 'caption1', 'caption2', 'caption3', 'caption4', 'caption5', 'CLC_codes', 'split']
All 10,000 features have attributes ✓


In [6]:
# ============================================================================
# WRITE GEOPACKAGE
# ============================================================================
# Remove any existing file so we start clean (GeoPackage appends layers by default)
if GPKG_OUT.exists():
    GPKG_OUT.unlink()

GPKG_OUT.parent.mkdir(parents=True, exist_ok=True)

merged.to_file(GPKG_OUT, layer=LAYER_NAME, driver="GPKG")

size_mb = GPKG_OUT.stat().st_size / (1024 * 1024)
print(f"✓ Wrote {GPKG_OUT}")
print(f"  Size : {size_mb:.2f} MB")
print(f"  Layer: {LAYER_NAME}")


✓ Wrote /home/ubuntu/SENSERO/sensero.gpkg
  Size : 8.61 MB
  Layer: patches


In [7]:
# ============================================================================
# ROUND-TRIP VERIFY
# ============================================================================
check = gpd.read_file(GPKG_OUT, layer=LAYER_NAME)

print(f"Re-read   : {len(check):,} features (expected {len(merged):,})")
print(f"CRS       : {check.crs}")
print(f"Columns   : {list(check.columns)}")
print()

# Confirm no truncation: longest caption preserved in full
max_len = check["caption3"].str.len().max()
print(f"Longest caption3 length: {max_len} chars "
      f"(would have been capped at 254 in DBF)")

# Spot-check a sample feature
sample = check.dropna(subset=[NEW_COLS[0]]).iloc[0]
print(f"\nSample feature ({sample['filename']}):")
for c in NEW_COLS:
    val = sample[c]
    if isinstance(val, str) and len(val) > 100:
        val = val[:97] + "..."
    print(f"  {c}: {val}")


Re-read   : 10,000 features (expected 10,000)
CRS       : EPSG:4326
Columns   : ['filename', 'rel_path', 'caption1', 'caption2', 'caption3', 'caption4', 'caption5', 'CLC_codes', 'split', 'geometry']

Longest caption3 length: 317 chars (would have been capped at 254 in DBF)

Sample feature (S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10056_3577):
  caption1: artificial surfaces, agricultural areas, forest and semi-natural zones
  caption2: industrial commercial and transport units, pastures, arable land, forests, heterogeneous agricult...
  caption3: road and rail networks and associated land, pastures, non-irrigated arable land, broad-leaved for...
  caption4: pastures, arable land, broad-leaved forest, complex cultivation patterns, urban fabric, land prin...
  caption5: pastures, agriculture, forests, mixed farmland, urban fabric
  CLC_codes: 211, 231, 311, 122, 242, 112, 243
  split: test
